# TriageOS 30k Trajectory SFT Training

This Colab notebook keeps the larger trajectory-build workflow, but avoids the earlier full-size 90k-style training run.
It generates a large trajectory dataset, samples it down to an exact 30k SFT dataset, then fine-tunes a 7B model with a conservative T4-friendly setup.

In [ ]:
!python -m pip install -e .[train,dev]
!python -m pip install -q unsloth trl datasets accelerate bitsandbytes

## Build Large Trajectory Dataset

Start with the larger trajectory generator so the training data includes multi-step workflow behavior.
This cell keeps the larger-build setup and writes a raw SFT dataset that we will downsample safely.

In [ ]:
!python -m support_triage_env.trajectory_dataset \
  --episodes-per-task 1000 \
  --seed 42 \
  --min-final-score 0.6 \
  --format sft \
  --output outputs/synthetic_trajectory_sft_large.jsonl

## Sample Exact 30k Dataset

This is the key change: keep the large-build generator, but train on an exact 30,000-row sample instead of the earlier oversized dataset.

In [ ]:
import json
import random
from pathlib import Path

source_path = Path("outputs/synthetic_trajectory_sft_large.jsonl")
target_path = Path("outputs/synthetic_trajectory_sft_30k.jsonl")
target_rows = 30_000

with source_path.open("r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

print("raw_rows:", len(rows))
if len(rows) < target_rows:
    raise ValueError(f"Need at least {target_rows} rows, but only found {len(rows)}")

random.Random(42).shuffle(rows)
sampled_rows = rows[:target_rows]

target_path.parent.mkdir(parents=True, exist_ok=True)
with target_path.open("w", encoding="utf-8") as f:
    for row in sampled_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("sampled_rows:", len(sampled_rows))
print("output:", target_path)

In [ ]:
from collections import Counter

task_counts = Counter(row["task_id"] for row in sampled_rows)
step_counts = Counter(row["step_index"] for row in sampled_rows)

print("Task distribution:")
print(task_counts)
print("\nStep distribution:")
print(step_counts)
print("\nExample row preview:\n")
print(sampled_rows[0]["text"][:1500])

## Load 7B Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

## Prepare Dataset Split

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {"text": row["text"], "task_id": row["task_id"]}
    for row in sampled_rows
]).train_test_split(test_size=0.02, seed=42)

train_dataset = dataset["train"].remove_columns(["task_id"])
eval_dataset = dataset["test"].remove_columns(["task_id"])

print(dataset)
print("train_rows:", len(train_dataset))
print("eval_rows:", len(eval_dataset))

## Train

This setup is intentionally conservative for T4: 30k rows, max sequence length 1024, batch size 1, gradient accumulation 8, and 1 epoch.
If memory is stable and training time is comfortable, you can later raise epochs to 2.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        output_dir="outputs/unsloth_triage_agent_7b_30k",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=1.5e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_steps=30,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)

In [ ]:
trainer_stats = trainer.train()
trainer_stats

## Save Adapter

In [ ]:
model.save_pretrained("outputs/unsloth_triage_agent_7b_30k")
tokenizer.save_pretrained("outputs/unsloth_triage_agent_7b_30k")